# Day 4 v2 — Model 04: multilingual-e5-small + DNN (Hướng A)

**Architecture:** `intfloat/multilingual-e5-small` (frozen, 118M params) → 384-dim dense embedding → PriceDNN head (6 ResidualBlocks)

**Tại sao thay MiniLM:** `paraphrase-multilingual-MiniLM-L12-v2` có limit 128 tokens — truncate âm thầm descriptions dài. `multilingual-e5-small` cùng dim (384) nhưng limit 512 tokens, VN-MTEB 60.66.

**Target:** MAE < 75k VND

**Dataset:** `SeanSunny/items_tv_v9` (train=269K, val=3926, test=3872)

## vast.ai Setup (chỉ chạy lần đầu)

```bash
pip install uv
uv sync
```

Restart kernel sau khi sync xong.

In [2]:
2

2

In [3]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..").resolve()))

import json
import torch

from pricer_vi_2.items import Item
from pricer_vi_2.evaluator import evaluate, plot_training_history
import pricer_vi_2.senttrans_model as sm

# Override encoder BEFORE creating runner
sm.ENCODER_NAME = "intfloat/multilingual-e5-small"

from pricer_vi_2.senttrans_model import SentTransRunner

print(f"Encoder: {sm.ENCODER_NAME}")
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.9.0+cu128).


Encoder: intfloat/multilingual-e5-small
CUDA: True
GPU: NVIDIA GeForce RTX 3090 Ti


## 1. Load Data

In [4]:
train, val, test = Item.from_hub("SeanSunny/items_tv_v9")
print(f"Train: {len(train):,} | Val: {len(val):,} | Test: {len(test):,}")

Train: 269,112 | Val: 3,926 | Test: 3,872


## 2. Pre-compute Embeddings

Encode 269K train + 3926 val với multilingual-e5-small (frozen, 512-token limit).
Lần đầu: ~10 phút trên GPU. Lần sau: load từ cache pkl.

In [5]:
runner = SentTransRunner(train, val)

cache_path = Path("cache/e5small_embeddings.pkl")
runner.encode_and_cache(cache_path=cache_path)

Loading encoder: intfloat/multilingual-e5-small


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

...

Encoding train embeddings (269K) — ~10 min on GPU...


Batches:   0%|          | 0/1052 [00:00<?, ?it/s]

Encoding val embeddings (3926)...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Embeddings cached to cache/e5small_embeddings.pkl
Train: torch.Size([269112, 384]) | Val: torch.Size([3926, 384])


## 3. Setup Model

DNN head: 6 ResidualBlocks, input_size=384 (embedding dim).

In [6]:
runner.setup(batch_size=256, num_blocks=6)

SentTrans DNN head: 203,063,297 trainable params | embedding_dim=384
Using cuda


## 4. Train

Max 15 epochs, early stopping patience=3. Val feedback dùng val[:1000] mỗi epoch.

In [7]:
history = runner.train(epochs=15, patience=3)

Epoch 1/15:   0%|          | 0/1052 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR

Epoch 1/15 | train_loss=1.1235 | val_loss=0.6335 | val_mae=151.19k | lr=0.000989
  ** best val_mae=151.19k


Epoch 2/15:   0%|          | 0/1052 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR

Epoch 2/15 | train_loss=0.6302 | val_loss=0.5710 | val_mae=141.78k | lr=0.000957
  ** best val_mae=141.78k


Epoch 3/15:   0%|          | 0/1052 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PAR

...

Epoch 13/15 | train_loss=0.2836 | val_loss=0.4346 | val_mae=108.09k | lr=0.000043


Epoch 14/15:   0%|          | 0/1052 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PA

Epoch 14/15 | train_loss=0.2711 | val_loss=0.4312 | val_mae=106.77k | lr=0.000011
  ** best val_mae=106.77k


Epoch 15/15:   0%|          | 0/1052 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PA

Epoch 15/15 | train_loss=0.2646 | val_loss=0.4300 | val_mae=106.23k | lr=0.000000
  ** best val_mae=106.23k


## 5. Training History

In [8]:
plot_training_history(history, title="multilingual-e5-small + DNN")

## 6. Save Weights + Val Predictions + Test Predictions

In [9]:
Path("weights").mkdir(exist_ok=True)
runner.save("weights/e5small_dnn.pth")
print("Saved weights/e5small_dnn.pth")

Path("val_predictions").mkdir(exist_ok=True)

print("Running val predictions (3926 samples)...")
val_preds = runner.val_predictions()
with open("val_predictions/e5small_val.json", "w") as f:
    json.dump(val_preds, f)

print("Encoding + predicting test set (3872 samples)...")
test_preds = runner.test_predictions(test)
with open("val_predictions/e5small_test.json", "w") as f:
    json.dump(test_preds, f)

print(f"Val: {len(val_preds)} | Test: {len(test_preds)}")

Saved weights/e5small_dnn.pth
Running val predictions (3926 samples)...


Encoding + predicting test set (3872 samples)...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Val: 3926 | Test: 3872


## 7. Evaluate on 200 Test Samples

In [10]:
def e5small_pricer(item):
    return runner.inference(item)

results = evaluate(e5small_pricer, test)
print(f"MAE: {results['mae']:.1f}k VND | MSE: {results['mse']:,.0f} | R2: {results['r2']:.1f}%")

  0%|          | 0/200 [00:00<?, ?it/s]

118 12 10 14 6 523 14 43 58 43 9 145 213 42 64 93 37 386 88 29 28 44 12 23 80 10 45 25 621 2 53 141 10 35 105 6 25 0 496 12 343 64 5 12 9 350 13 4 22 41 168 111 152 413 205 110 183 51 165 485 133 101 30 66 100 86 2 38 315 13 16 20 12 140 0 43 16 36 157 27 150 487 171 14 98 198 70 3 219 40 33 18 163 602 15 11 30 120 3 4 62 131 33 59 38 26 30 29 15 19 65 32 239 255 157 19 160 82 29 6 205 46 68 32 74 6 16 15 233 14 274 1 232 1 103 12 293 27 45 81 179 40 2 3 263 2 149 41 49 6 402 488 205 421 31 14 97 115 22 15 12 136 85 598 52 44 3 188 6 238 130 31 108 88 34 86 146 0 162 113 180 56 5 32 47 52 3 365 699 28 153 99 33 1 21 23 62 180 8 2 


MAE: 102.7k VND | MSE: 28,526 | R2: 50.3%


## 8. Sanity Check — Load Roundtrip

In [11]:
sample = test[0]
pred_original = runner.inference(sample)
print(f"Product: {sample.title[:60]}")
print(f"Actual:  {sample.price:.1f}k VND")
print(f"Predict: {pred_original:.1f}k VND")
print(f"Error:   {abs(pred_original - sample.price):.1f}k VND")

runner.load("weights/e5small_dnn.pth")
pred_loaded = runner.inference(sample)
diff = abs(pred_original - pred_loaded)
assert diff < 0.01, f"Load mismatch: before={pred_original:.2f} after={pred_loaded:.2f}"
print(f"\nLoad roundtrip PASSED. Diff: {diff:.4f}k")

Product: Thùng lưu trữ, hộp đựng đồ đa năng bằng nhựa PP cao cấp 30L 
Actual:  479.0k VND
Predict: 597.2k VND
Error:   118.2k VND

Load roundtrip PASSED. Diff: 0.0000k


In [ ]:
import gc, os, subprocess, torch

# 1. Free PyTorch cache của session hiện tại
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# 2. Kill zombie processes đang giữ VRAM (không kill process hiện tại)
current_pid = os.getpid()
result = subprocess.run(
    ["nvidia-smi", "--query-compute-apps=pid", "--format=csv,noheader"],
    capture_output=True, text=True
)
killed = []
for line in result.stdout.strip().splitlines():
    try:
        pid = int(line.strip())
        if pid != current_pid:
            os.kill(pid, 9)
            killed.append(pid)
    except (ValueError, ProcessLookupError):
        pass
print(f"Killed PIDs: {killed}" if killed else "No zombie GPU processes.")

# 3. Restart kernel (KHONG restart may)
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)